# Datasets de IA · OpenAlex — enfoque agregado

En lugar de descargar cada paper individualmente (costoso en requests y tiempo), se generan **tres datasets complementarios** con ~400 requests totales (~5 min):

| Dataset | Archivo | Método API | Responde |
|---------|---------|------------|----------|
| **A** — conteos por topic y año | `ai_topic_year_counts.csv` | `group_by=primary_topic.id` | Q1 · Q2 · Q3 |
| **B** — conteos por país y año | `ai_geo_year_counts.csv` | `group_by=country_code` | Q4 |
| **C** — muestra de papers | `ai_papers_sample.csv` | papers individuales, 500/año todos los años | Q5 · Q6 · Q7 |

**Alcance:** subfields *Artificial Intelligence* (1702) y *Computer Vision & Pattern Recognition* (1707), 1950–hoy.

In [1]:
import requests
import pandas as pd
import time
import datetime
import os
import pycountry  # Librería para convertir códigos ISO en nombres de países

# Configuración
API_KEY     = "" # Opcional: tu API Key de OpenAlex
BASE_URL    = "https://api.openalex.org/works"
HEADERS     = {"User-Agent": "DataVizProject/1.0 (mailto:tu@email.com)"}

# Filtro: Subfields de Inteligencia Artificial (AI)
# 1702: Artificial Intelligence
# 1707: Computer Vision and Pattern Recognition
AI_SUBFIELD_IDS = ["1702", "1707"]
SUBFIELD_FILTER = "|".join(AI_SUBFIELD_IDS)

# Parámetros de tiempo y volumen
START_YEAR      = 1950
END_YEAR        = 2024
SAMPLE_PER_YEAR = 1000

today = datetime.date.today()
print(f"Configuración lista para AI (Subfields: {SUBFIELD_FILTER})")

Configuración lista para AI (Subfields: 1702|1707)


In [2]:
# ── Helper: rango de fechas de un año ────────────────────────────────────────
def year_dates(year):
    from_d = f"{year}-01-01"
    to_d   = f"{year}-12-31" if year < today.year else today.strftime("%Y-%m-%d")
    return from_d, to_d

# ── Helper: group_by para un año concreto ─────────────────────────────────────
def groupby_year(year, group_field):
    from_d, to_d = year_dates(year)
    resp = requests.get(BASE_URL, headers=HEADERS, params={
        "filter"  : (
            f"primary_topic.subfield.id:{SUBFIELD_FILTER},"
            f"type:article,"
            f"from_publication_date:{from_d},"
            f"to_publication_date:{to_d}"
        ),
        "group_by": group_field,
        "per_page": 200,
        "api_key" : API_KEY,
    })
    resp.raise_for_status()
    data = resp.json()
    time.sleep(0.2)
    return data["group_by"], data["meta"]["count"]   # grupos + total real del año

# ── Metadata de países (mapeo ISO → Nombre) ──────────────────────────────────
# Usamos pycountry como fuente primaria y la API de OpenAlex como respaldo/limpieza
def get_country_name(iso_code):
    try:
        # pycountry usa códigos de 2 letras (alpha_2)
        country = pycountry.countries.get(alpha_2=iso_code.upper())
        if country:
            return country.name
    except:
        pass
    return f"Unknown ({iso_code})"

print("Cargando nombres de países (usando pycountry)...")
# Prueba rápida
print(f"Prueba: US -> {get_country_name('US')}")
print(f"Prueba: ES -> {get_country_name('ES')}")
print("Helpers definidos.")

Cargando nombres de países (usando pycountry)...
Prueba: US -> United States
Prueba: ES -> Spain
Helpers definidos.


In [3]:
# ── Metadata de topics (una sola llamada) ─────────────────────────────────────
# Necesaria para enriquecer el group_by con subfield/field/domain
print("Cargando metadata de topics de IA...")
topic_meta = {}
cursor = "*"
while cursor:
    resp = requests.get(
        "https://api.openalex.org/topics",
        headers=HEADERS,
        params={
            "filter"  : f"subfield.id:{SUBFIELD_FILTER}",
            "per_page": 200,
            "cursor"  : cursor,
            "api_key" : API_KEY,
        },
    )
    resp.raise_for_status()
    data = resp.json()
    for t in data["results"]:
        topic_meta[t["id"]] = {
            "topic_id"     : t["id"],
            "topic_name"   : t["display_name"],
            "subfield_id"  : t["subfield"]["id"],
            "subfield_name": t["subfield"]["display_name"],
            "field_id"     : t["field"]["id"],
            "field_name"   : t["field"]["display_name"],
            "domain_id"    : t["domain"]["id"],
            "domain_name"  : t["domain"]["display_name"],
        }
    cursor = data["meta"].get("next_cursor")
    time.sleep(0.2)

print(f"Topics cargados: {len(topic_meta)}")

Cargando metadata de topics de IA...
Topics cargados: 112


## Dataset A — Conteos por topic y año
**→ `ai_topic_year_counts.csv`**  
Responde Q1 (tendencias), Q2 (crecimiento), Q3 (popularidad).  
Método: `group_by=primary_topic.id` · 1 request/año · conteos reales, sin muestreo.

In [4]:
rows_A = []
t0     = time.time()

for year in range(START_YEAR, END_YEAR + 1):
    if year > today.year:
        break

    groups, total = groupby_year(year, "primary_topic.id")

    for g in groups:
        meta = topic_meta.get(g["key"], {})
        rows_A.append({
            "year"         : year,
            "total_papers" : total,           # total real del año (sin muestreo)
            "topic_id"     : g["key"],
            "topic_name"   : g["key_display_name"],
            "subfield_id"  : meta.get("subfield_id"),
            "subfield_name": meta.get("subfield_name"),
            "field_id"     : meta.get("field_id"),
            "field_name"   : meta.get("field_name"),
            "domain_id"    : meta.get("domain_id"),
            "domain_name"  : meta.get("domain_name"),
            "n_papers"     : g["count"],
        })

    print(f"  {year}  total={total:>7,}  topics={len(groups):>3}  [{time.time()-t0:.0f}s]")

df_A = pd.DataFrame(rows_A)
print(f"\nDataset A: {len(df_A):,} filas  ({df_A['year'].min()}–{df_A['year'].max()})")

  1950  total=  1,409  topics= 88  [1s]
  1951  total=  1,545  topics= 86  [2s]
  1952  total=  1,614  topics= 89  [3s]
  1953  total=  1,764  topics= 86  [4s]
  1954  total=  1,808  topics= 86  [4s]
  1955  total=  1,878  topics= 88  [5s]
  1956  total=  1,966  topics= 92  [6s]
  1957  total=  2,056  topics= 93  [6s]
  1958  total=  2,211  topics= 97  [7s]
  1959  total=  2,516  topics= 96  [8s]
  1960  total=  2,735  topics= 95  [9s]
  1961  total=  2,871  topics= 95  [9s]
  1962  total=  2,999  topics=100  [10s]
  1963  total=  3,435  topics= 96  [11s]
  1964  total=  3,789  topics=106  [12s]
  1965  total=  4,137  topics=104  [12s]
  1966  total=  4,371  topics= 99  [13s]
  1967  total=  4,772  topics=106  [14s]
  1968  total=  5,344  topics=102  [15s]
  1969  total=  6,016  topics=106  [15s]
  1970  total=  6,831  topics=107  [16s]
  1971  total=  6,309  topics=108  [17s]
  1972  total=  6,744  topics=107  [18s]
  1973  total=  6,955  topics=107  [19s]
  1974  total=  7,375  topic

## Dataset B — Conteos por país y año
**→ `ai_geo_year_counts.csv`**  
Responde Q4 (distribución geográfica).  
Método: `group_by=authorships.institutions.country_code` · 1 request/año · conteos reales.

In [4]:
rows_B = []
t0     = time.time()

for year in range(START_YEAR, END_YEAR + 1):
    if year > today.year:
        break

    groups, total = groupby_year(year, "authorships.institutions.country_code")

    for g in groups:
        if not g["key"]:   # saltar entradas sin country_code
            continue
        
        # Limpiar la URL completa para obtener solo el código ISO (ej: 'US')
        iso_code = g["key"].split("/")[-1].upper()
        
        rows_B.append({
            "year"         : year,
            "total_papers" : total,
            "country_code" : iso_code,
            "country_name" : get_country_name(iso_code),  # Usamos la nueva función con pycountry
            "n_papers"     : g["count"],
        })

    print(f"  {year}  países={len([g for g in groups if g['key']]):>3}  [{time.time()-t0:.0f}s]")

df_B = pd.DataFrame(rows_B)
print(f"\nDataset B: {len(df_B):,} filas")

  1950  países= 31  [1s]
  1951  países= 31  [2s]
  1952  países= 31  [2s]
  1953  países= 38  [3s]
  1954  países= 39  [4s]
  1955  países= 38  [4s]
  1956  países= 34  [5s]
  1957  países= 44  [5s]
  1958  países= 37  [6s]
  1959  países= 45  [7s]
  1960  países= 43  [7s]
  1961  países= 46  [8s]
  1962  países= 48  [9s]
  1963  países= 45  [9s]
  1964  países= 51  [10s]
  1965  países= 48  [10s]
  1966  países= 63  [11s]
  1967  países= 58  [12s]
  1968  países= 53  [12s]
  1969  países= 68  [13s]
  1970  países= 74  [14s]
  1971  países= 61  [14s]
  1972  países= 63  [15s]
  1973  países= 69  [16s]
  1974  países= 72  [16s]
  1975  países= 73  [17s]
  1976  países= 75  [18s]
  1977  países= 72  [18s]
  1978  países= 71  [19s]
  1979  países= 76  [20s]
  1980  países= 72  [20s]
  1981  países= 79  [21s]
  1982  países= 85  [21s]
  1983  países= 84  [22s]
  1984  países= 88  [23s]
  1985  países= 88  [23s]
  1986  países= 92  [24s]
  1987  países= 88  [25s]
  1988  países= 99  [25s]


## Dataset C — Muestra de papers individuales
**→ `ai_papers_sample.csv`**  
Responde Q5 (publicaciones ↔ citas), Q6 (co-autoría internacional ↔ citas), Q7 (Open Access ↔ citas).  
Método: papers individuales · todos los años → 1000/año ordenados por fecha (muestra no sesgada).  
Estadísticamente, 1000 observaciones/año es un buen balance entre costo y precisión para análisis de proporciones y correlaciones.

In [6]:
SELECT_C = ["id", "publication_date", "primary_topic",
            "cited_by_count", "open_access", "authorships"]

def extract_paper(work: dict) -> dict:
    pt  = work.get("primary_topic") or {}
    oa  = work.get("open_access")   or {}
    aus = work.get("authorships")   or []
    cc_set = {
        inst.get("country_code")
        for auth in aus
        for inst in (auth.get("institutions") or [])
        if inst.get("country_code")
    }
    unique_cc = sorted(cc_set)
    pub_date  = work.get("publication_date") or ""
    year      = int(pub_date[:4]) if len(pub_date) >= 4 else None
    return {
        "work_id"          : work.get("id"),
        "publication_date" : pub_date,
        "year"             : year,
        "primary_topic_id" : pt.get("id"),
        "primary_topic"    : pt.get("display_name"),
        "subfield_id"      : (pt.get("subfield") or {}).get("id"),
        "subfield_name"    : (pt.get("subfield") or {}).get("display_name"),
        "cited_by_count"   : work.get("cited_by_count", 0),
        "oa_status"        : oa.get("oa_status"),
        "is_oa"            : oa.get("is_oa", False),
        "n_authors"        : len(aus),
        "n_countries"      : len(unique_cc),
        "is_international" : len(unique_cc) > 1,
        "countries"        : ";".join(unique_cc),
    }

rows_C = []
t0     = time.time()

for year in range(START_YEAR, END_YEAR + 1):
    if year > today.year:
        break

    from_d, to_d = year_dates(year)
    n_max        = SAMPLE_PER_YEAR

    query = (
        Works()
        .filter(
            primary_topic={"subfield": {"id": SUBFIELD_FILTER}},
            type="article",
            from_publication_date=from_d,
            to_publication_date=to_d,
        )
        .sort(publication_date="asc")   # orden por fecha → muestra no sesgada por relevancia
        .select(SELECT_C)
    )

    n_year = 0
    for page in query.paginate(per_page=200, n_max=n_max):
        for work in page:
            rows_C.append(extract_paper(work))
            n_year += 1
        time.sleep(0.12)

    print(f"  {year}  n={n_year:>4}  [{time.time()-t0:.0f}s]")

df_C = pd.DataFrame(rows_C)
print(f"\nDataset C: {len(df_C):,} filas")

  1950  n=1000  [4s]
  1951  n=1000  [9s]
  1952  n=1000  [13s]
  1953  n=1000  [19s]
  1954  n=1000  [23s]
  1955  n=1000  [28s]
  1956  n=1000  [33s]
  1957  n=1000  [37s]
  1958  n=1000  [42s]
  1959  n=1000  [46s]
  1960  n=1000  [50s]
  1961  n=1000  [54s]
  1962  n=1000  [58s]
  1963  n=1000  [63s]
  1964  n=1000  [67s]
  1965  n=1000  [71s]
  1966  n=1000  [76s]
  1967  n=1000  [80s]
  1968  n=1000  [85s]
  1969  n=1000  [89s]
  1970  n=1000  [93s]
  1971  n=1000  [98s]
  1972  n=1000  [102s]
  1973  n=1000  [106s]
  1974  n=1000  [110s]
  1975  n=1000  [115s]
  1976  n=1000  [120s]
  1977  n=1000  [124s]
  1978  n=1000  [128s]
  1979  n=1000  [133s]
  1980  n=1000  [138s]
  1981  n=1000  [143s]
  1982  n=1000  [147s]
  1983  n=1000  [151s]
  1984  n=1000  [156s]
  1985  n=1000  [161s]
  1986  n=1000  [165s]
  1987  n=1000  [169s]
  1988  n=1000  [174s]
  1989  n=1000  [180s]
  1990  n=1000  [185s]
  1991  n=1000  [193s]
  1992  n=1000  [199s]
  1993  n=1000  [205s]
  1994  n=10

In [5]:
df_B.to_csv("ai_geo_year_counts.csv",   index=False)

In [7]:
# ── Guardar los tres datasets ─────────────────────────────────────────────────
df_A.to_csv("ai_topic_year_counts.csv", index=False)
df_B.to_csv("ai_geo_year_counts.csv",   index=False)
df_C.to_csv("ai_papers_sample.csv",     index=False)

SEP = "─" * 55
print(SEP)
print("  ARCHIVOS GENERADOS")
print(SEP)

for label, df, fname in [
    ("Dataset A — topic/año",   df_A, "ai_topic_year_counts.csv"),
    ("Dataset B — país/año",    df_B, "ai_geo_year_counts.csv"),
    ("Dataset C — muestra",     df_C, "ai_papers_sample.csv"),
]:
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  {fname}")
    print(f"    {len(df):>8,} filas × {len(df.columns)} cols  (~{mb:.1f} MB)")

print()
print("Dataset A · papers reales por año (muestra del primer/último año):")
for yr in [START_YEAR, 1970, 1990, 2000, 2010, 2020, END_YEAR]:
    row = df_A[df_A["year"] == yr]
    if not row.empty:
        total = row["total_papers"].iloc[0]
        print(f"  {yr}: {total:>9,} papers reales")

print()
print("Dataset C · OA status:")
for status, cnt in df_C["oa_status"].value_counts(dropna=False).items():
    print(f"  {str(status):<10}  {cnt:>6,}  ({cnt/len(df_C):.1%})")

print()
print(f"Dataset C · papers internacionales : {df_C['is_international'].mean():.1%}")
print(f"Dataset C · mediana citas          : {df_C['cited_by_count'].median():.0f}")
print(f"Dataset C · media citas            : {df_C['cited_by_count'].mean():.1f}")

───────────────────────────────────────────────────────
  ARCHIVOS GENERADOS
───────────────────────────────────────────────────────
  ai_topic_year_counts.csv
       8,206 filas × 11 cols  (~5.0 MB)
  ai_geo_year_counts.csv
       8,044 filas × 4 cols  (~0.8 MB)
  ai_papers_sample.csv
      77,000 filas × 14 cols  (~44.0 MB)

Dataset A · papers reales por año (muestra del primer/último año):
  1950:     1,409 papers reales
  1970:     6,831 papers reales
  1990:    21,059 papers reales
  2000:    42,868 papers reales
  2010:   131,105 papers reales
  2020:   167,732 papers reales
  2026:   118,497 papers reales

Dataset C · OA status:
  closed      52,900  (68.7%)
  gold        10,143  (13.2%)
  green        6,025  (7.8%)
  bronze       4,151  (5.4%)
  diamond      2,641  (3.4%)
  hybrid       1,140  (1.5%)

Dataset C · papers internacionales : 10.5%
Dataset C · mediana citas          : 36
Dataset C · media citas            : 112.6
